# 🚀 HUẤN LUYỆN REAL TEXT MANIPULATION (RTM) TRÊN GOOGLE COLAB A100

> **Chuẩn bị trước khi bắt đầu:**
> 1. Vào menu **Runtime** -> **Change runtime type** -> Chọn **A100 GPU**.
> 2. Đảm bảo file `RealTextMan.zip` (2.4 GB) đã được tải lên Google Drive của bạn.

### Bước 1: Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone Repository từ GitHub (Nhánh `experiment/prototype-contrastive`)

In [ ]:
%cd /content
!rm -rf TMD
!git clone -b experiment/prototype-contrastive https://github.com/gagaduy/TMD.git
%cd /content/TMD/ASCFormer

### Bước 3: Copy và Giải nén Dataset từ Google Drive vào SSD Colab
> *Lưu ý: Copy vào đĩa `/content/` giúp tốc độ đọc ảnh nhanh gấp 15-20 lần so với đọc trực tiếp qua Drive.*

In [ ]:
# Tạo thư mục đích
!mkdir -p /content/TMD/ASCFormer/data/ttd

# Tự động tìm file RealTextMan.zip trên Google Drive
import os, shutil, glob

drive_zip = '/content/drive/MyDrive/RealTextMan.zip'
if not os.path.exists(drive_zip):
    matches = glob.glob('/content/drive/MyDrive/**/RealTextMan.zip', recursive=True)
    if matches:
        drive_zip = matches[0]
        print(f'Tìm thấy file zip tại: {drive_zip}')
    else:
        raise FileNotFoundError('Không tìm thấy file RealTextMan.zip trên Google Drive! Hãy kiểm tra lại vị trí file.')

print('Đang copy dataset từ Drive sang ổ cứng Colab SSD...')
shutil.copy(drive_zip, '/content/TMD/ASCFormer/data/ttd/RealTextMan.zip')

print('Đang giải nén dataset...')
%cd /content/TMD/ASCFormer/data/ttd
!unzip -q RealTextMan.zip
!rm RealTextMan.zip
print('✅ Giải nén hoàn tất! Kiểm tra thư mục:')
!ls -lh RealTextMan/

### Bước 4: Cài đặt Thư viện (Đã chuẩn hóa phiên bản - Triệt tiêu 100% xung đột)
> *Nguyên nhân gây lỗi trên Colab: OpenMMLab 2.x bị xung đột nếu cài `mmcv` biên dịch CUDA mới nhất hoặc `mmengine > 0.8.0`. Sử dụng các phiên bản ghim cố định dưới đây sẽ chạy ngay lập tức.*

In [ ]:
# 1. Gỡ bỏ các gói xung đột nếu có
!pip uninstall -y mmcv mmcv-lite mmengine mmsegmentation timm yapf > /dev/null 2>&1

# 2. Cài đặt các phiên bản tương thích tuyệt đối cho A100 & Colab
!pip install -q "numpy<2.0.0"
!pip install -q mmcv-lite==2.0.0
!pip install -q mmengine==0.7.0
!pip install -q timm==0.4.12 einops==0.6.1

# 3. Vá triệt để lỗi lib2to3 của Python 3.13 (Dummy yapf FormatCode)
import site, os
for p in site.getsitepackages():
    yapf_dir = os.path.join(p, 'yapf', 'yapflib')
    os.makedirs(yapf_dir, exist_ok=True)
    with open(os.path.join(p, 'yapf', '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, '__init__.py'), 'w') as f: pass
    with open(os.path.join(yapf_dir, 'yapf_api.py'), 'w') as f:
        f.write('def FormatCode(text, **kwargs):
    return text, True
')

# 4. Đăng ký ASCFormer vào Python Path trực tiếp
for p in site.getsitepackages():
    try:
        with open(os.path.join(p, 'ascformer.pth'), 'w') as f:
            f.write('/content/TMD/ASCFormer
')
        break
    except Exception:
        pass

%env PYTHONPATH=/content/TMD/ASCFormer:

# 5. Tạo thư mục .mim hỗ trợ OpenMMLab
!mkdir -p /content/TMD/ASCFormer/mmseg/.mim
!ln -sf /content/TMD/ASCFormer/tools /content/TMD/ASCFormer/mmseg/.mim/tools
!ln -sf /content/TMD/ASCFormer/configs /content/TMD/ASCFormer/mmseg/.mim/configs

print('✅ Môi trường & Bản vá Python 3.13 đã hoàn tất thành công!')

### Bước 5: Kiểm tra Môi trường (Sanity Check)

In [ ]:
import torch
import mmcv
import mmengine
import mmseg

print('=' * 45)
print('PyTorch Version :', torch.__version__)
print('CUDA Sẵn sàng   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Model       :', torch.cuda.get_device_name(0))
    print('VRAM Khả dụng   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
print('MMCV-Lite       :', mmcv.__version__)
print('MMEngine        :', mmengine.__version__)
print('MMSegmentation  :', mmseg.__version__)
print('=' * 45)
print('🚀 SẴN SÀNG HUẤN LUYỆN TRÊN A100!')

### Bước 6: Khởi động Huấn luyện Mô hình Đóng góp 3 trên A100-80GB (< 1 Giờ)
> **Cấu hình High-Capacity tối ưu cho A100-80GB:**
> * `batch_size = 4`, `num_workers = 8` (nạp 32,000 mẫu ảnh — gấp đôi pilot cũ)
> * `max_iters = 8000` (đủ 8,000 bước cập nhật gradient cho Dual-Prototypes hội tụ)
> * `lr = 6e-5` (chuẩn ổn định, triệt tiêu hiện tượng sốc gradient ở Decode Head)
> * `val_dataloader.num_workers = 8` (tối ưu hóa thời gian validation)
> * **Thời gian hoàn thành:** ~50-55 phút (< 1 giờ), kỳ vọng IoU $\ge 2.0\% - 2.5\%$!


In [ ]:
%cd /content/TMD/ASCFormer

# Cấu hình tối ưu toàn diện cho A100 80GB (8,000 iters, batch_size 4, ~50 phút):
!python tools/train.py configs/ascformer/ascformer_rtm_b0_tristream_proto_a100_80gb.py


### Bước 7: Sao lưu Checkpoints & Logs về Google Drive
> *Sau khi train xong, chạy ô này để lưu vĩnh viễn trọng số (.pth) và log vào thư mục `MyDrive/RTM_Checkpoints/` trên Drive của bạn.*

In [ ]:
!mkdir -p /content/drive/MyDrive/RTM_Checkpoints
!cp -r /content/TMD/ASCFormer/work_dirs/* /content/drive/MyDrive/RTM_Checkpoints/
print("✅ Đã sao lưu thành công checkpoints và logs về Google Drive!")
